# Uncovering Academic Profiles via Dimensionality Reduction
## Notebook 1: Exploratory Data Analysis & Linear Dimensionality Reduction (PCA)

### Problem Definition
Raw academic grade averages frequently obscure subtle inter-disciplinary trade-offs—such as strong performance in STEM subjects paired with moderate struggle in Humanities, or high elective scores masking core academic risk.

This notebook applies **Principal Component Analysis (PCA)** to an 8-dimensional student grade dataset (`math`, `science`, `cs`, `band`, `english`, `history`, `spanish`, `physed`) to:
1. Audit and explore the raw multi-subject grade structure.
2. Standardize features so subject-level variance differences do not distort principal axes.
3. Reduce $\mathbb{R}^8$ grade vectors into $\mathbb{R}^2$ and $\mathbb{R}^3$ subspaces while retaining 85–96% of total variance.
4. Interpret latent academic axes via **Component Loadings (Eigenvectors)**.
5. Profile distinct student archetypes via **K-Means Clustering** to support data-driven academic advising and targeted interventions.

---
## 1. Import Necessary Libraries
We import standard scientific Python libraries for data manipulation (`pandas`, `numpy`), visualization (`matplotlib`, `seaborn`, `plotly`), preprocessing (`StandardScaler`), linear dimensionality reduction (`PCA`), and clustering (`KMeans`).

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px

from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans

# Set consistent visual style and random seed for reproducibility
sns.set_theme(style="whitegrid", context="notebook")
RANDOM_STATE = 42

---
## 2. Data Import & Initial Inspection (EDA)
We load `student_grades.csv` and audit its shape, data types, missing values, duplicates, and summary statistics. Notice that `student_id` is a unique identifier and must be isolated from the 8 numeric subject features prior to modeling.

In [ ]:
# Load dataset
df = pd.read_csv("student_grades.csv")

# Define the 8 core subject columns (excluding student_id)
SUBJECTS = ["math", "science", "cs", "band", "english", "history", "spanish", "physed"]

print(f"Dataset Shape: {df.shape[0]} students x {df.shape[1]} columns")
print("\n--- Data Info ---")
df.info()

print("\n--- Missing Values per Column ---")
print(df.isnull().sum())

print(f"\nDuplicate Rows: {df.duplicated().sum()}")
df.head()

In [ ]:
# Summary statistics across all 8 subjects
summary_stats = df[SUBJECTS].describe().T[["count", "mean", "std", "min", "25%", "50%", "75%", "max"]].round(2)
print("--- Subject Grade Summary Statistics ---")
summary_stats

### 2.1 Distribution & Correlation Analysis
Visualizing subject distributions and pairwise correlations reveals two critical facts before we run PCA:
* **Scale / Variance Disparity:** Humanities subjects (`english`, `history`, `spanish`) have standard deviations of **17.3–18.4**, whereas electives (`band`: 6.28, `physed`: 8.07) are tightly clustered. Without standardization, Humanities would dominate PC1 solely due to larger raw variance.
* **Block Structure:** Three distinct correlation blocks exist—**STEM** (`0.97–0.98`), **Humanities** (`0.96–0.97`), and **Electives** (`0.80`, almost orthogonal to core academics).

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 5.5))

# Left: Subject Grade Distributions (Boxplot)
sns.boxplot(data=df[SUBJECTS], palette="Set2", ax=axes[0])
axes[0].set_title("Distribution of Raw Student Grades (8 Subjects)", fontsize=13, fontweight="bold")
axes[0].set_xlabel("Subject")
axes[0].set_ylabel("Raw Grade Score")
axes[0].tick_params(axis="x", rotation=45)

# Right: Pairwise Correlation Heatmap
corr_matrix = df[SUBJECTS].corr()
sns.heatmap(
    corr_matrix,
    annot=True,
    fmt=".2f",
    cmap="coolwarm",
    center=0,
    vmin=-1,
    vmax=1,
    linewidths=0.5,
    ax=axes[1]
)
axes[1].set_title("Subject Grade Correlation Matrix", fontsize=13, fontweight="bold")

plt.tight_layout()
plt.show()

---
## 3. Data Preprocessing & Feature Standardization
PCA identifies orthogonal axes that maximize variance. To ensure every subject contributes equally regardless of its raw spread, we isolate `student_id` and apply **Z-score Standardization** (`StandardScaler`):

$$z = \frac{x - \mu}{\sigma}$$

After scaling, every subject has mean $\mu = 0.0$ and standard deviation $\sigma = 1.0$.

In [ ]:
# Isolate student_id from numeric feature matrix
student_ids = df["student_id"]
X_raw = df[SUBJECTS].copy()

# Apply Z-score standardization
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_raw)
df_scaled = pd.DataFrame(X_scaled, columns=SUBJECTS)

# Verify before vs. after mean and standard deviation
verification_df = pd.DataFrame({
    "Original Mean": X_raw.mean().round(2),
    "Original Std": X_raw.std(ddof=0).round(2),
    "Scaled Mean": df_scaled.mean().round(4),
    "Scaled Std": df_scaled.std(ddof=0).round(4)
})
print("--- Standardization Verification ---")
verification_df

---
## 4. Linear Dimensionality Reduction — Full PCA
We fit PCA across all 8 components first to inspect eigenvalues, individual explained variance ratios, and cumulative explained variance. Our target is **85–95% variance coverage** using the fewest possible dimensions.

In [ ]:
# Fit full PCA across all 8 dimensions
pca_full = PCA(random_state=RANDOM_STATE)
X_pca_full = pca_full.fit_transform(X_scaled)

explained_var_pct = pca_full.explained_variance_ratio_ * 100
cumulative_var_pct = np.cumsum(explained_var_pct)

variance_table = pd.DataFrame({
    "Component": [f"PC{i}" for i in range(1, len(SUBJECTS) + 1)],
    "Eigenvalue": pca_full.explained_variance_.round(3),
    "Variance Explained (%)": explained_var_pct.round(2),
    "Cumulative Variance (%)": cumulative_var_pct.round(2)
})
print("--- PCA Explained Variance Table ---")
variance_table

In [ ]:
# Scree Plot & Cumulative Explained Variance Chart
fig, axes = plt.subplots(1, 2, figsize=(13, 4.8))
components_range = range(1, len(SUBJECTS) + 1)

# Scree Plot
axes[0].plot(components_range, explained_var_pct, marker="o", linewidth=2.2, color="#1f77b4")
for i, val in enumerate(explained_var_pct):
    axes[0].text(i + 1, val + 1.8, f"{val:.1f}%", ha="center", fontsize=9)
axes[0].set_title("Scree Plot (Variance per Principal Component)", fontsize=12, fontweight="bold")
axes[0].set_xlabel("Principal Component")
axes[0].set_ylabel("% Variance Explained")
axes[0].set_xticks(components_range)
axes[0].set_ylim(0, max(explained_var_pct) + 12)

# Cumulative Explained Variance
axes[1].plot(components_range, cumulative_var_pct, marker="s", linewidth=2.2, color="teal")
axes[1].axhline(85, color="red", linestyle="--", linewidth=1.3, label="85% Target")
axes[1].axhline(95, color="orange", linestyle=":", linewidth=1.5, label="95% Target")
for i, val in enumerate(cumulative_var_pct):
    offset = -5.5 if val > 92 else 2.5
    axes[1].text(i + 1, val + offset, f"{val:.1f}%", ha="center", fontsize=9)
axes[1].set_title("Cumulative Explained Variance", fontsize=12, fontweight="bold")
axes[1].set_xlabel("Number of Principal Components")
axes[1].set_ylabel("Cumulative % Variance")
axes[1].set_xticks(components_range)
axes[1].set_ylim(0, 108)
axes[1].legend(loc="lower right")

plt.tight_layout()
plt.show()

### Key Finding from Variance Analysis
* **2 Components (`PC1 + PC2`)** capture **87.79%** of total variance (already exceeding the 85% benchmark).
* **3 Components (`PC1 + PC2 + PC3`)** capture **96.00%** of total variance (exceeding the 95% benchmark).
* Components `PC4` through `PC8` together account for only **4.00%** and largely represent minor within-subject noise.

---
## 5. Latent Feature Interpretation & Component Loadings
To assign semantic meaning to `PC1`, `PC2`, and `PC3`, we inspect the PCA eigenvector weights (component loadings).

In [ ]:
# Extract loadings for the top 3 principal components
loadings_df = pd.DataFrame(
    pca_full.components_[:3].T,
    columns=["PC1 (Core Aptitude)", "PC2 (Electives)", "PC3 (STEM vs Humanities)"],
    index=SUBJECTS
)
print("--- PCA Component Loadings (PC1 - PC3) ---")
print(loadings_df.round(3))

# Plot Component Loadings Bar Chart
ax = loadings_df.plot(
    kind="bar",
    figsize=(11, 5),
    width=0.75,
    color=["#1f77b4", "#ff7f0e", "#2ca02c"]
)
plt.axhline(0, color="black", linewidth=0.9)
plt.title("PCA Component Loadings (PC1, PC2, PC3) by Subject", fontsize=13, fontweight="bold")
plt.xlabel("Subject")
plt.ylabel("Eigenvector Loading Weight")
plt.xticks(rotation=45)
plt.ylim(-0.85, 0.85)
plt.legend(title="Principal Component")
plt.tight_layout()
plt.show()

### Semantic Interpretation of Latent Axes
1. **PC1 (65.20% variance) — Overall Core Academic Aptitude:** All 6 core subjects (`math`, `science`, `cs`, `english`, `history`, `spanish`) load almost identically around **+0.41**, while electives (`band`, `physed`) are near zero. High PC1 indicates strong overall academic achievement; low PC1 flags at-risk students.
2. **PC2 (22.59% variance) — Electives Orientation:** Driven almost entirely by `band` (**+0.706**) and `physed` (**+0.702**).
3. **PC3 (8.21% variance) — STEM vs. Humanities Trade-off:** Separates STEM subjects with negative weights (`math`: -0.397, `science`: -0.382, `cs`: -0.437) from Humanities subjects with positive weights (`english`: +0.412, `history`: +0.397, `spanish`: +0.411).

---
## 6. 2D & 3D PCA Projections with K-Means Student Profiles
We apply **K-Means (`k=4`)** on the standardized 8-subject data to identify 4 consistent academic clusters, then project them into both **2D** (`PC1 vs PC2`, `PC1 vs PC3`) and **3D** (`PC1 vs PC2 vs PC3`, covering 96% of variance).

In [ ]:
# Assign 4 clusters using K-Means on the standardized feature space
kmeans = KMeans(n_clusters=4, random_state=RANDOM_STATE, n_init=10)
df["Cluster"] = kmeans.fit_predict(X_scaled)

# Map numeric cluster IDs to human-readable academic profiles (verified in Section 7)
PROFILE_MAP = {
    0: "Humanities-Focused",
    1: "Humanities & Electives Leaders",
    2: "At-Risk Students",
    3: "High-Achieving All-Rounders"
}
df["Profile"] = df["Cluster"].map(PROFILE_MAP)

# Attach top 3 PC coordinates to DataFrame
df["PC1"] = X_pca_full[:, 0]
df["PC2"] = X_pca_full[:, 1]
df["PC3"] = X_pca_full[:, 2]

# 2D PCA Scatter Plots: (PC1 vs PC2) and (PC1 vs PC3)
fig, axes = plt.subplots(1, 2, figsize=(15, 5.5))

sns.scatterplot(
    data=df, x="PC1", y="PC2", hue="Profile",
    palette="viridis", s=70, alpha=0.9, ax=axes[0]
)
axes[0].set_title("2D PCA: PC1 (Core Aptitude) vs. PC2 (Electives)", fontsize=12, fontweight="bold")
axes[0].set_xlabel(f"PC1 ({explained_var_pct[0]:.1f}% var)")
axes[0].set_ylabel(f"PC2 ({explained_var_pct[1]:.1f}% var)")

sns.scatterplot(
    data=df, x="PC1", y="PC3", hue="Profile",
    palette="viridis", s=70, alpha=0.9, ax=axes[1]
)
axes[1].set_title("2D PCA: PC1 (Core Aptitude) vs. PC3 (STEM vs. Humanities)", fontsize=12, fontweight="bold")
axes[1].set_xlabel(f"PC1 ({explained_var_pct[0]:.1f}% var)")
axes[1].set_ylabel(f"PC3 ({explained_var_pct[2]:.1f}% var)")

plt.tight_layout()
plt.show()

In [ ]:
# Static 3D PCA Scatter Plot (Matplotlib — captures 96.0% of total variance)
fig = plt.figure(figsize=(10, 7))
ax = fig.add_subplot(111, projection="3d")

palette = sns.color_palette("viridis", n_colors=4)
for cluster_id, profile_name in PROFILE_MAP.items():
    subset = df[df["Cluster"] == cluster_id]
    ax.scatter(
        subset["PC1"], subset["PC2"], subset["PC3"],
        label=f"{profile_name} (n={len(subset)})",
        color=palette[cluster_id],
        s=65, alpha=0.9, edgecolors="w", linewidth=0.5
    )

ax.set_title("3D PCA Projection (96.0% Cumulative Explained Variance)", fontsize=13, fontweight="bold", pad=15)
ax.set_xlabel(f"PC1: Core Aptitude ({explained_var_pct[0]:.1f}%)")
ax.set_ylabel(f"PC2: Electives ({explained_var_pct[1]:.1f}%)")
ax.set_zlabel(f"PC3: STEM vs Humanities ({explained_var_pct[2]:.1f}%)")
ax.legend(loc="upper left", bbox_to_anchor=(0.0, 0.95))
plt.tight_layout()
plt.show()

In [ ]:
# Interactive 3D PCA Scatter Plot (Plotly — hover over points to inspect individual students)
fig_3d = px.scatter_3d(
    df,
    x="PC1", y="PC2", z="PC3",
    color="Profile",
    hover_data=["student_id"] + SUBJECTS,
    title="Interactive 3D PCA Student Academic Profiles (96.0% Variance)",
    labels={
        "PC1": f"PC1: Core Aptitude ({explained_var_pct[0]:.1f}%)",
        "PC2": f"PC2: Electives ({explained_var_pct[1]:.1f}%)",
        "PC3": f"PC3: STEM vs Humanities ({explained_var_pct[2]:.1f}%)"
    }
)
fig_3d.update_traces(marker=dict(size=6, line=dict(width=0.5, color="white")))
fig_3d.show()

---
## 7. Cluster Profiling & Business Recommendations
We aggregate the raw subject grades by cluster to validate our profile labels and translate findings into actionable advising strategies.

In [ ]:
# Compute mean grades and domain averages per profile
profile_summary = df.groupby("Profile")[SUBJECTS].mean().round(1)
profile_summary["Count"] = df.groupby("Profile")["student_id"].count()
profile_summary["STEM_Avg"] = df.groupby("Profile")[["math", "science", "cs"]].mean().mean(axis=1).round(1)
profile_summary["Hum_Avg"] = df.groupby("Profile")[["english", "history", "spanish"]].mean().mean(axis=1).round(1)
profile_summary["Elec_Avg"] = df.groupby("Profile")[["band", "physed"]].mean().mean(axis=1).round(1)
profile_summary["Overall_Avg"] = df.groupby("Profile")[SUBJECTS].mean().mean(axis=1).round(1)

print("--- Academic Profile Summary Table ---")
print(profile_summary)

# Heatmap of average subject grades by profile
plt.figure(figsize=(10, 4.8))
sns.heatmap(
    profile_summary[SUBJECTS],
    annot=True, fmt=".1f", cmap="YlGnBu", linewidths=0.5,
    cbar_kws={"label": "Mean Grade"}
)
plt.title("Mean Subject Grades by Identified Academic Profile", fontsize=13, fontweight="bold")
plt.xlabel("Subject")
plt.ylabel("Academic Profile")
plt.yticks(rotation=0)
plt.tight_layout()
plt.show()

### Actionable Recommendations for Educators & Advisors
| Academic Profile | Share | Key Data Insight | Recommended Intervention |
| :--- | :---: | :--- | :--- |
| **At-Risk Students** | 44% | Average in electives (`68.4`) but critically low across all 6 core subjects (`42–48`). | **Immediate Intervention & Targeted Tutoring:** Prioritize foundational tutoring in Math, Science, and English with bi-weekly advisor check-ins. |
| **Humanities-Focused** | 25% | Strong Humanities (`74.1`) paired with a ~19-point STEM deficit (`55.0`) and lower electives (`63.1`). | **Data-Driven Advising:** Pair humanities-track advising with targeted STEM bridge workshops to close the quantitative gap. |
| **Humanities & Electives Leaders** | 22% | Highest scores in `band` (`76.4`) and `physed` (`76.2`), solid Humanities (`73.4`), moderate STEM (`58.0`). | **Engagement-Leveraged Support:** Use high extracurricular engagement to form peer-led STEM study groups. |
| **High-Achieving All-Rounders** | 9% | Elite performance across both STEM (`86.1`) and Humanities (`90.5`). | **Enrichment & Mentorship:** Recommend Honors/AP tracks, interdisciplinary capstone projects, and peer-tutoring leadership roles. |

---
*Next Step: Proceed to `02_tSNE_Analysis.ipynb` to evaluate non-linear manifold learning (t-SNE & UMAP), tune `perplexity` and `learning_rate`, and benchmark non-linear embeddings against PCA.*